In [147]:
from sage.all import*

In [148]:
def Pivot(A, i, W):
    m = A.ncols()
    I = -1
    D = -1
    DW = -1  # shifted pivot degree

    for j in range(m):
        if A[i,j] != 0:
            if A[i,j].degree() + W[j] >= DW: 
                DW = A[i,j].degree()+W[j]
                D = A[i,j].degree()
                I = j

    return (I, D, DW)

In [149]:
def is_reduced(A,W=0):
    n, m = A.nrows(), A.ncols()

    # If no weights are given, set W = [0,0,...,0]
    if W == 0:
        W=zero_vector(ZZ, A.ncols())
        
    piv = [Pivot(A,i,W) for i in range(n)]  # (I)

    # gather nonzero rows
    nz = [i for i in range(n) if not A[i].is_zero()]

    # distinct pivot columns among nonzero rows
    cols = [piv[i][0] for i in nz]
    if len(set(cols)) != len(cols):
        return False

    return True

In [150]:
def SimpleTransformation(A,U,i,j,I,d_i,d_j):
    m = A.ncols()
    d = d_i-d_j

    cxe = (A[i,I].leading_coefficient() / A[j,I].leading_coefficient())*(x**d)
    for l in range(m):
        A[i,l] = A[i,l] - cxe*A[j,l]
        U[i,l] = U[i,l] - cxe*U[j,l]

    return (A, U)

In [151]:
def WeightedDegreeOfMatrix(A,W):
    n = A.nrows()
    m=A.ncols()
    max_deg = -oo

    for i in range(n):
        for j in range(m):
            max_deg = max(A[i,j].degree()+W[j], max_deg)
    return max_deg

In [152]:
def WeightedRowDegree(A, i, W):
    m = A.ncols()
    degs = []

    for j in range(m):
        if A[i, j] != 0:
            degs.append(A[i, j].degree() + W[j])

    if len(degs) == 0:
        return -oo

    return max(degs)

In [153]:
def WeightedSumDegreeOfMatrix(A, W):
    n = A.nrows()
    DegreeSum = 0

    for i in range(n):
        row_deg = WeightedRowDegree(A, i, W)
        if row_deg != -oo:
            DegreeSum += row_deg
    
    return DegreeSum

In [154]:
def BaseStep(A,R,W=0):
    # If no weights are given, set W = [0,0,...,0]
    if W == 0:
        W=zero_vector(ZZ, A.ncols())
        
    D_old = WeightedSumDegreeOfMatrix(A,W)
    D_new = oo
    n = A.nrows()
    U = matrix.identity(R, n, n)
    
    
    pivots = zero_matrix(ZZ, n, 2)

    for i in range(n):
        pivots[i,0] = Pivot(A,i,W)[0] 
        pivots[i,1] = Pivot(A,i,W)[1]

    while D_old <= D_new:

        if is_reduced(A):
            return U

        broke = False

        for i in range(n):
            for j in range(i+1,n):
                if pivots[i,0] != -1 and pivots[j,0] != -1 and pivots[i,0] == pivots[j,0]:
                    if pivots[i,1] >= pivots[j,1]:
                        D_old = WeightedSumDegreeOfMatrix(A, W)
                        A, U = SimpleTransformation(A,U,i,j,pivots[i,0],pivots[i,1],pivots[j,1])
                        D_new = WeightedSumDegreeOfMatrix(A,W)
                        pivots[i,0] = Pivot(A,i,W)[0]
                        pivots[i,1] = Pivot(A,i,W)[1]
                    else:
                        D_old = WeightedSumDegreeOfMatrix(A, W)
                        A, U = SimpleTransformation(A,U,j,i,pivots[j,0],pivots[j,1],pivots[i,1])
                        D_new = WeightedSumDegreeOfMatrix(A,W)
                        pivots[j,0] = Pivot(A,j,W)[0]
                        pivots[j,1] = Pivot(A,j,W)[1]
                    broke = True
                    break
            if broke:
                break
                        
    return U

In [155]:
def AccuracyApproximation(A, t, W=0):
    n = A.nrows()
    m = A.ncols()

    if W == 0:
        W = zero_vector(ZZ, m)

    R = A.base_ring()
    x = R.gen()

    A_approx = zero_matrix(R, n, m)

    for i in range(n):
        # Weighted row degree of row i
        row_degree = -oo
        for j in range(m):
            if A[i, j] != 0:
                row_degree = max(row_degree, A[i, j].degree() + W[j])

        if row_degree == -oo:
            continue

        cutoff = row_degree - t

        for j in range(m):
            p = A[i, j]

            if p == 0:
                continue

            q = R.zero()

            for exponent, coefficient in p.dict().items():
                # Sage sometimes stores univariate exponents as integers,
                # and sometimes as one-tuples.
                if isinstance(exponent, tuple):
                    exponent = exponent[0]

                if exponent + W[j] > cutoff:
                    q += coefficient * x**exponent

            A_approx[i, j] = q

    return A_approx

In [156]:
def Alekhnovich(A, R, t, W=0):
    if W == 0:
        W = zero_vector(ZZ, A.ncols())

    t = int(t)

    if t <= 1:
        return BaseStep(A, R, W)

    A_approx = AccuracyApproximation(A, t, W)

    if is_reduced(A_approx, W):
        return identity_matrix(R, A.nrows())

    U_prime = Alekhnovich(A_approx, R, floor(t / 2), W)
    A_prime = U_prime * A_approx

    if is_reduced(A_prime, W):
        return U_prime

    degree_drop = WeightedSumDegreeOfMatrix(A_approx, W) - WeightedSumDegreeOfMatrix(A_prime, W)
    remaining_t = t - degree_drop

    # Safety fallback: if the recursive step made no progress, finish using
    # the direct row-reduction routine instead of recursing forever.
    if degree_drop <= 0 or remaining_t >= t:
        return BaseStep(A_prime, R, W) * U_prime

    if remaining_t <= 1:
        return BaseStep(A_prime, R, W) * U_prime

    return Alekhnovich(A_prime, R, remaining_t, W) * U_prime

In [157]:
import time
import random as pyrandom
import statistics as stats

import Weakpopov as weak_module


def random_poly(R, max_degree, density=0.9):
    x = R.gen()

    if pyrandom.random() > density:
        return R.zero()

    d = ZZ.random_element(0, max_degree + 1)
    coeffs = [R.base_ring().random_element() for _ in range(d + 1)]

    while d > 0 and coeffs[d] == 0:
        coeffs[d] = R.base_ring().random_element()

    return sum(coeffs[i] * x**i for i in range(d + 1))


def random_matrix(R, rows, cols, max_degree, density=0.9):
    return Matrix(R, rows, cols, [
        random_poly(R, max_degree, density)
        for _ in range(rows * cols)
    ])

def hard_random_matrix(R, n, max_degree, density=0.9):
    x = R.gen()
    A = random_matrix(R, n, n, max_degree, density)

    for i in range(n):
        A[i, n-1] = random_poly(R, max_degree + n + i, density=1.0)

    return A


def time_weak_popov(A, W, max_seconds=3600):
    B = copy(A)

    start = time.perf_counter()
    weak_module.WeakPopov(B, W)
    elapsed = time.perf_counter() - start

    if elapsed > max_seconds:
        return None

    return elapsed


def time_alekhnovich(A, R, W, max_seconds=3600):
    B = copy(A)
    t = WeightedSumDegreeOfMatrix(B, W) + 1

    start = time.perf_counter()
    Alekhnovich(B, R, t, W)
    elapsed = time.perf_counter() - start

    if elapsed > max_seconds:
        return None

    return elapsed


def run_simulations(
    sizes=(2, 3, 4, 5, 6),
    max_degrees=(2, 3, 4, 5),
    field_size=7,
    density=0.9,
    trials=5,
    seed=42,
):
    set_random_seed(seed)
    pyrandom.seed(int(seed))

    F = GF(field_size)
    R = PolynomialRing(F, "x")
    x = R.gen()

    globals()["R"] = R
    globals()["x"] = x

    weak_module.x = x

    results = []

    for n in sizes:
        for max_degree in max_degrees:
            W = zero_vector(ZZ, n)

            weak_times = []
            alek_times = []

            for _ in range(trials):
                A = random_matrix(R, n, n, max_degree, density)

                weak_time = time_weak_popov(A, W)
                alek_time = time_alekhnovich(A, R, W)

                if weak_time is not None:
                    weak_times.append(weak_time)

                if alek_time is not None:
                    alek_times.append(alek_time)
                
            if len(weak_times) == 0 or len(alek_times) == 0:
                print(f"n={n}, degree<={max_degree}: skipped because all runs timed out")
                continue

            result = {
                "n": n,
                "max_degree": max_degree,
                "field_size": field_size,
                "density": density,
                "trials": trials,
                "WeakPopov_mean": stats.mean(weak_times),
                "WeakPopov_median": stats.median(weak_times),
                "Alekhnovich_mean": stats.mean(alek_times),
                "Alekhnovich_median": stats.median(alek_times),
            }

            results.append(result)

            print(
                f"n={n}, degree<={max_degree}: "
                f"WeakPopov median={result['WeakPopov_median']:.6f}s, "
                f"Alekhnovich median={result['Alekhnovich_median']:.6f}s"
            )

    return results

In [158]:
import math


def loglog(d):
    return math.log(max(math.log(d), 1.000001))


def weakpopov_complexity_term(row):
    n = row["n"]
    d = row["max_degree"]

    # If you do not measure r separately, set r = 1.
    # Then this checks whether runtime roughly follows n^2 d^2.
    r = row.get("r", 1)

    return n**2 * r * d**2


def alekhnovich_complexity_term(row):
    n = row["n"]
    d = row["max_degree"]

    if d <= 1:
        return n**2 * d

    return n**3 * d * (math.log(d)**2) * loglog(d) + n**2 * d


def add_normalized_times(results):
    normalized = []

    for row in results:
        weak_term = weakpopov_complexity_term(row)
        alek_term = alekhnovich_complexity_term(row)

        new_row = dict(row)
        new_row["WeakPopov_normalized"] = row["WeakPopov_median"] / weak_term
        new_row["Alekhnovich_normalized"] = row["Alekhnovich_median"] / alek_term

        normalized.append(new_row)

    return normalized


normalized_results = add_normalized_times(results)

for row in normalized_results:
    print(
        f"n={row['n']}, d={row['max_degree']}: "
        f"Weak norm={row['WeakPopov_normalized']:.3e}, "
        f"Alek norm={row['Alekhnovich_normalized']:.3e}"
    )

n=5, d=5: Weak norm=1.011e-06, Alek norm=1.173e-05
n=5, d=10: Weak norm=4.789e-07, Alek norm=1.755e-06
n=5, d=15: Weak norm=1.063e-07, Alek norm=1.137e-06
n=5, d=20: Weak norm=6.310e-08, Alek norm=4.568e-07
n=5, d=30: Weak norm=3.478e-08, Alek norm=1.859e-07
n=5, d=40: Weak norm=3.656e-08, Alek norm=8.134e-07
n=5, d=50: Weak norm=4.526e-08, Alek norm=1.079e-07
n=5, d=60: Weak norm=3.118e-08, Alek norm=8.719e-08
n=5, d=80: Weak norm=5.020e-09, Alek norm=8.367e-08
n=5, d=100: Weak norm=5.923e-09, Alek norm=2.640e-07
n=10, d=5: Weak norm=8.340e-07, Alek norm=2.076e-06
n=10, d=10: Weak norm=1.892e-07, Alek norm=3.618e-07
n=10, d=15: Weak norm=2.320e-07, Alek norm=3.740e-07
n=10, d=20: Weak norm=4.817e-08, Alek norm=1.947e-07
n=10, d=30: Weak norm=8.387e-08, Alek norm=6.827e-08
n=10, d=40: Weak norm=2.818e-08, Alek norm=5.131e-08
n=10, d=50: Weak norm=2.095e-08, Alek norm=4.122e-08
n=10, d=60: Weak norm=2.074e-08, Alek norm=3.115e-08
n=10, d=80: Weak norm=3.459e-08, Alek norm=2.852e-08
n=10

In [ ]:
results = run_simulations(
    sizes=(50,100,150,200,250,300,350,400,450,500,550,600,650,700,750,800,850,900,950,1000),
    max_degrees=(50,100,150,200,250,300,350,400,450,500,550,600,650,700,750,800,850,900,950,1000),
    trials=3,
)

n=5, degree<=5: WeakPopov median=0.000702s, Alekhnovich median=0.028159s
n=5, degree<=10: WeakPopov median=0.000772s, Alekhnovich median=0.041760s
n=5, degree<=15: WeakPopov median=0.000851s, Alekhnovich median=0.101009s
n=5, degree<=20: WeakPopov median=0.000826s, Alekhnovich median=0.056678s
n=5, degree<=30: WeakPopov median=0.000803s, Alekhnovich median=0.070524s
n=5, degree<=40: WeakPopov median=0.001069s, Alekhnovich median=0.105008s
n=5, degree<=50: WeakPopov median=0.002311s, Alekhnovich median=0.193037s
n=5, degree<=60: WeakPopov median=0.002906s, Alekhnovich median=0.253200s
n=5, degree<=70: WeakPopov median=0.003577s, Alekhnovich median=0.297799s
n=5, degree<=80: WeakPopov median=0.000548s, Alekhnovich median=0.071972s
n=5, degree<=90: WeakPopov median=0.003204s, Alekhnovich median=0.312134s
n=5, degree<=100: WeakPopov median=0.004627s, Alekhnovich median=0.591223s
n=5, degree<=110: WeakPopov median=0.004962s, Alekhnovich median=0.643514s
n=5, degree<=120: WeakPopov median=0.

In [161]:
for row in results:
    print(row)

{'n': 5, 'max_degree': 5, 'field_size': 7, 'density': 0.900000000000000, 'trials': 3, 'WeakPopov_mean': 0.0012619796664997314, 'WeakPopov_median': 0.0009681670053396374, 'Alekhnovich_mean': 0.031490521005859286, 'Alekhnovich_median': 0.027617512008873746}
{'n': 5, 'max_degree': 10, 'field_size': 7, 'density': 0.900000000000000, 'trials': 3, 'WeakPopov_mean': 0.0011385836648211505, 'WeakPopov_median': 0.0005948429898126051, 'Alekhnovich_mean': 0.037618728000476644, 'Alekhnovich_median': 0.03443412799970247}
{'n': 5, 'max_degree': 15, 'field_size': 7, 'density': 0.900000000000000, 'trials': 3, 'WeakPopov_mean': 0.0006398636711916575, 'WeakPopov_median': 0.0006462450110120699, 'Alekhnovich_mean': 0.06120011266708995, 'Alekhnovich_median': 0.05906648399832193}
{'n': 5, 'max_degree': 20, 'field_size': 7, 'density': 0.900000000000000, 'trials': 3, 'WeakPopov_mean': 0.0009124463346476356, 'WeakPopov_median': 0.0012030070065520704, 'Alekhnovich_mean': 0.05512906800140627, 'Alekhnovich_median':

In [162]:
import math


def fit_power_law_one_variable(results, algorithm, variable="n"):
    xs = []
    ys = []

    time_key = algorithm + "_median"

    for row in results:
        if row[time_key] > 0:
            xs.append(float(row[variable]))
            ys.append(float(row[time_key]))

    log_xs = [math.log(x) for x in xs]
    log_ys = [math.log(y) for y in ys]

    mean_x = sum(log_xs) / len(log_xs)
    mean_y = sum(log_ys) / len(log_ys)

    numerator = sum((x - mean_x) * (y - mean_y) for x, y in zip(log_xs, log_ys))
    denominator = sum((x - mean_x)**2 for x in log_xs)

    alpha = numerator / denominator
    log_C = mean_y - alpha * mean_x
    C = math.exp(log_C)

    return {
        "algorithm": algorithm,
        "variable": variable,
        "C": C,
        "exponent": alpha,
        "model": f"T ≈ {C:.3e} * {variable}^{alpha:.3f}",
    }


def fit_power_law_two_variables(results, algorithm):
    time_key = algorithm + "_median"

    rows = [
        row for row in results
        if row[time_key] > 0 and row["n"] > 0 and row["max_degree"] > 0
    ]

    # Model:
    # log(T) = log(C) + alpha log(n) + beta log(d)
    X = []
    y = []

    for row in rows:
        X.append([
            1.0,
            math.log(float(row["n"])),
            math.log(float(row["max_degree"])),
        ])
        y.append(math.log(float(row[time_key])))

    M = matrix(RR, X)
    b = vector(RR, y)

    coeffs = (M.transpose() * M).inverse() * M.transpose() * b

    log_C = coeffs[0]
    alpha = coeffs[1]
    beta = coeffs[2]
    C = math.exp(log_C)

    return {
        "algorithm": algorithm,
        "C": C,
        "n_exponent": alpha,
        "degree_exponent": beta,
        "model": f"T ≈ {C:.3e} * n^{alpha:.3f} * d^{beta:.3f}",
    }

In [163]:
fit_power_law_one_variable(results, "WeakPopov", variable="n")

{'algorithm': 'WeakPopov',
 'variable': 'n',
 'C': 4.217979949331064e-05,
 'exponent': 2.2751538393372166,
 'model': 'T ≈ 4.218e-05 * n^2.275'}

In [164]:
fit_power_law_one_variable(results, "Alekhnovich", variable="n")

{'algorithm': 'Alekhnovich',
 'variable': 'n',
 'C': 0.0052140237360980704,
 'exponent': 2.1713290961417293,
 'model': 'T ≈ 5.214e-03 * n^2.171'}

In [165]:
fit_power_law_two_variables(results, "WeakPopov")

{'algorithm': 'WeakPopov',
 'C': 2.7544657378145215e-06,
 'n_exponent': 2.45086864216968,
 'degree_exponent': 0.596206500919169,
 'model': 'T ≈ 2.754e-06 * n^2.451 * d^0.596'}

In [166]:
fit_power_law_two_variables(results, "Alekhnovich")

{'algorithm': 'Alekhnovich',
 'C': 0.00011148815007313561,
 'n_exponent': 2.41893863950825,
 'degree_exponent': 0.840147882051043,
 'model': 'T ≈ 1.115e-04 * n^2.419 * d^0.840'}